# Steam Reviews Analysis
This notebook works only with the original dataset. Running it is not recommended as it takes a while. Use the parquet file of the sampled data directly from the repo instead.
Sections cover stratified sampling and verification of the sampled distribution.

In [2]:
from google.colab import drive
import pandas as pd
import numpy as np
import os

drive.mount('/content/drive')
path = '/content/drive/MyDrive/steam_reviews_mining'
file_path = os.path.join(path, 'steam_reviews.csv')
print('Environment configured and dataset path set.')


Mounted at /content/drive
Environment configured and dataset path set.


In [ ]:
chunk = pd.read_csv(file_path, nrows=5)
chunk.head()


,Unnamed: 0,app_id,app_name,review_id,language,review,timestamp_created,timestamp_updated,recommended,votes_helpful,...,steam_purchase,received_for_free,written_during_early_access,author.steamid,author.num_games_owned,author.num_reviews,author.playtime_forever,author.playtime_last_two_weeks,author.playtime_at_review,author.last_played
0,0,292030,The Witcher 3: Wild Hunt,85185598,schinese,不玩此生遗憾，RPG游戏里的天花板，太吸引人了,1611381629,1611381629,True,0,...,True,False,False,76561199095369542,6,2,1909.0,1448.0,1909.0,1.611343e+09
1,1,292030,The Witcher 3: Wild Hunt,85185250,schinese,拔DIAO无情打桩机--杰洛特!!!,1611381030,1611381030,True,0,...,True,False,False,76561198949504115,30,10,2764.0,2743.0,2674.0,1.611386e+09
2,2,292030,The Witcher 3: Wild Hunt,85185111,schinese,巫师3NB,1611380800,1611380800,True,0,...,True,False,False,76561199090098988,5,1,1061.0,1061.0,1060.0,1.611384e+09
3,3,292030,The Witcher 3: Wild Hunt,85184605,english,"One of the best RPG's of all time, worthy of a...",1611379970,1611379970,True,0,...,True,False,False,76561199054755373,5,3,5587.0,3200.0,5524.0,1.611384e+09
4,4,292030,The Witcher 3: Wild Hunt,85184287,schinese,大作,1611379427,1611379427,True,0,...,True,False,False,76561199028326951,7,4,217.0,42.0,217.0,1.610788e+09


## Stratified Sampling
The dataset is large, so we read it in chunks and sample 12% of each `app_id` group to keep a representative subset.


In [4]:
sampled_chunks = []
for chunk in pd.read_csv(file_path, chunksize=500_000):
    sample = chunk.groupby('app_id').sample(frac=0.12, random_state=42)
    sampled_chunks.append(sample)

df = pd.concat(sampled_chunks, ignore_index=True)
print(f'Sampled DataFrame rows: {len(df):,}')


Sampled DataFrame rows: 2,609,677


## Verification
Compare app-level counts between the original dataset and the sample to confirm the sampling ratio is stable.


In [5]:
original_counts = pd.Series(dtype='int64')
for chunk in pd.read_csv(file_path, usecols=['app_id'], chunksize=500_000):
    counts = chunk['app_id'].value_counts()
    original_counts = original_counts.add(counts, fill_value=0)

sampled_counts = df['app_id'].value_counts()
comparison = pd.DataFrame({
    'original': original_counts,
    'sampled': sampled_counts,
    'ratio': sampled_counts / original_counts
})
comparison_sorted = comparison.sort_values('original', ascending=False)
print('Top 15 games by original volume (comparison table):')
print(comparison_sorted.head(15))
print(f"Mean sampling ratio: {comparison['ratio'].mean():.4f}")


Top 15 games by original volume (comparison table):
         original  sampled     ratio
app_id                              
578080  1644255.0   197311  0.120000
271590  1019116.0   122294  0.120000
359550   841918.0   101031  0.120001
105600   672815.0    80738  0.120000
4000     655524.0    78663  0.120000
252490   549074.0    65889  0.120000
252950   498565.0    59828  0.120000
218620   487747.0    58529  0.119999
945360   485293.0    58235  0.120000
292030   469395.0    56327  0.119999
381210   418897.0    50268  0.120001
346110   400009.0    48001  0.120000
227300   387553.0    46506  0.119999
413150   315717.0    37886  0.120000
72850    294966.0    35396  0.120000
Mean sampling ratio: 0.1200


In [6]:
df.to_parquet(os.path.join(path, 'steam_reviews_sampled.parquet'), index=False)
print(f'Saved {len(df):,} rows')

Saved 2,609,677 rows
